# SnowHealth Demand Forecasting — Hyperparameter Tuning

Uses the **Snowflake ML Tuner API** to run distributed hyperparameter optimization on `SNOWHEALTH_COMPUTE_POOL`. The Tuner orchestrates parallel trials via Ray Tune on Snowpark Container Services — each trial trains an XGBoost model with a different configuration, and the best model is registered in the Model Registry.

| Step | What happens |
|---|---|
| Prepare data | Write train/test splits to Snowflake tables for `DataConnector` access |
| Define search | Search space over 5 XGBoost hyperparameters using `RandomSearch` |
| Run Tuner | 10 trials on `SNOWHEALTH_COMPUTE_POOL` via Ray Tune |
| Evaluate | Compare baseline vs best trial on held-out test set |
| Register | Add `tuned_v1` to Model Registry as the new default |

> **Requires:** `03_model_training.ipynb` run first. This notebook must be connected to a **Container Runtime** service backed by `SNOWHEALTH_COMPUTE_POOL`.

### Initialize Session
Connects to Snowflake, loads config and shared utilities. Sets the session to the `SNOWHEALTH_MODELS` schema where the Model Registry lives.

In [ ]:
import sys, os

# Resolve source/ (works from root or notebooks/ subdirectory)
for _p in [os.path.join(os.getcwd(), 'source'), os.path.join(os.getcwd(), '..', 'source')]:
    if os.path.isdir(_p): sys.path.insert(0, os.path.realpath(_p)); break

from snowflake.snowpark.context import get_active_session
import config as cfg
from model_utils import (
    train_and_evaluate, compute_metrics, get_feature_importance,
    generate_comparison_report,
)
from feature_engineering import get_feature_columns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_MODELS_SCHEMA}").collect()
print(f"Session ready: {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}")

### Prepare Data — Train/Test Splits as Snowflake Tables
The Snowflake ML Tuner uses `DataConnector` to stream data to each trial worker. `DataConnector.from_dataframe()` reads from a Snowpark DataFrame (backed by a Snowflake table), so we write the train/test splits to temporary tables. This ensures each trial worker gets the same data without transferring DataFrames over the wire.

In [ ]:
# Load features and split (same seed as notebook 03 for consistency)
df = session.table(cfg.SNOWHEALTH_FEATURE_TABLE).to_pandas()
feature_cols = get_feature_columns(df, cfg.TARGET_COLUMN)

X = df[feature_cols].astype(float)
y = df[cfg.TARGET_COLUMN].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE)

# Write splits to Snowflake tables for DataConnector
train_df = X_train.copy()
train_df[cfg.TARGET_COLUMN] = y_train.values
test_df = X_test.copy()
test_df[cfg.TARGET_COLUMN] = y_test.values

train_table = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.HPO_TRAIN_DATA"
test_table = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.HPO_TEST_DATA"

session.create_dataframe(train_df).write.mode('overwrite').save_as_table(train_table)
session.create_dataframe(test_df).write.mode('overwrite').save_as_table(test_table)

print(f"Train: {len(train_df):,} rows → {train_table}")
print(f"Test:  {len(test_df):,} rows → {test_table}")
print(f"Features: {len(feature_cols)}")

### Train Baseline for Comparison
Re-trains the baseline XGBoost locally so we have an apples-to-apples comparison point — we don't rely on notebook 03's metrics since the feature set may have changed.

In [ ]:
from xgboost import XGBRegressor

xgb_baseline, baseline_preds, baseline_metrics = train_and_evaluate(
    XGBRegressor, cfg.XGBOOST_BASELINE_PARAMS,
    X_train, y_train, X_test, y_test,
    fit_kwargs={'eval_set': [(X_test, y_test)], 'verbose': False},
)

print('Baseline XGBoost:')
for k, v in baseline_metrics.items():
    print(f'  {k}: {v}')

### Define Training Function for the Tuner
The Tuner runs each trial as an isolated function on a **remote Ray worker**. Because workers don't have access to the workspace filesystem, the function must be self-contained — it cannot import from `source/model_utils.py`. All dependencies (XGBoost, sklearn, numpy) must be imported inside the function.

Inside each trial:
1. `get_tuner_context()` provides the sampled hyperparameters and data connectors
2. The function trains an XGBoost model with those params
3. `ctx.report()` sends the RMSE metric and trained model back to the Tuner

The metric name (`rmse`) must match the `TunerConfig.metric` field exactly.

> **Note:** `model_utils.compute_rmse()` provides this same calculation for notebook-side code. The duplication inside `train_func` is intentional — remote workers can't reach local files.

In [ ]:
from snowflake.ml.modeling.tune import get_tuner_context

def train_func():
    """Self-contained training function for each Tuner trial (runs on remote Ray workers)."""
    from xgboost import XGBRegressor
    from sklearn.metrics import mean_squared_error
    import numpy as np

    ctx = get_tuner_context()
    params = ctx.get_hyper_params()
    datasets = ctx.get_dataset_map()

    train_pdf = datasets['train'].to_pandas()
    test_pdf = datasets['test'].to_pandas()

    target = 'UNITS_SOLD'
    X_tr = train_pdf.drop(columns=[target])
    y_tr = train_pdf[target]
    X_te = test_pdf.drop(columns=[target])
    y_te = test_pdf[target]

    model = XGBRegressor(
        n_estimators=int(params['n_estimators']),
        max_depth=int(params['max_depth']),
        learning_rate=params['learning_rate'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        objective='reg:squarederror',
        tree_method='hist',
        random_state=42,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

    preds = model.predict(X_te)
    rmse = float(np.sqrt(mean_squared_error(y_te, preds)))
    ctx.report(metrics={'rmse': rmse}, model=model)

print("Training function defined")

### Define Search Space and Tuner Configuration
The search space uses Snowflake ML's sampling functions (`randint`, `uniform`, `loguniform`) which are compatible with Ray Tune under the hood.

| Parameter | Range | Type | Why |
|---|---|---|---|
| `n_estimators` | 100–500 | Integer | Number of boosting rounds |
| `max_depth` | 3–10 | Integer | Tree complexity |
| `learning_rate` | 0.01–0.3 | Log-uniform | Step size (log-scale samples more in the 0.01–0.1 range) |
| `subsample` | 0.6–1.0 | Uniform | Row sampling per tree |
| `colsample_bytree` | 0.6–1.0 | Uniform | Feature sampling per tree |

`RandomSearch` with 10 trials gives a good balance of exploration vs runtime for a demo.

In [ ]:
from snowflake.ml.modeling.tune import Tuner, TunerConfig, randint, uniform, loguniform
from snowflake.ml.modeling.tune.search import RandomSearch

search_space = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 10),
    'learning_rate': loguniform(0.01, 0.3),
    'subsample': uniform(0.6, 1.0),
    'colsample_bytree': uniform(0.6, 1.0),
}

tuner_config = TunerConfig(
    metric='rmse',
    mode='min',
    search_alg=RandomSearch(),
    num_trials=10,
    uses_snowflake_trainer=False,
)

print(f"Search space: {list(search_space.keys())}")
print(f"Trials: {tuner_config.num_trials} | Metric: {tuner_config.metric} | Mode: {tuner_config.mode}")

### Run Distributed HPO on SNOWHEALTH_COMPUTE_POOL
The `Tuner.run()` call submits the training function to the compute pool. Each trial runs as an isolated Ray worker — up to `max_nodes` trials can run in parallel. The Tuner handles scheduling, fault tolerance, and result collection.

This cell may take 3–5 minutes depending on compute pool provisioning and trial execution time.

In [ ]:
from snowflake.ml.data.data_connector import DataConnector

train_connector = DataConnector.from_dataframe(session.table(train_table))
test_connector = DataConnector.from_dataframe(session.table(test_table))

tuner = Tuner(train_func, search_space, tuner_config)
print(f"Starting HPO: 10 trials on {cfg.SNOWHEALTH_COMPUTE_POOL}...")
results = tuner.run(dataset_map={'train': train_connector, 'test': test_connector})

print(f"\nAll trials:")
print(results.results[['config/n_estimators', 'config/max_depth', 'config/learning_rate', 'rmse']].sort_values('rmse').to_string(index=False))

print(f"\nBest trial RMSE: {results.best_result['rmse'].iloc[0]:.4f}")

### Evaluate Best Model vs Baseline
The Tuner saves the best model object (passed via `ctx.report(model=model)`). We evaluate it on the same held-out test set and compare against the baseline using `generate_comparison_report()`.

In [ ]:
best_model = results.best_model
tuned_preds = best_model.predict(X_test)
tuned_metrics = compute_metrics(y_test.values, tuned_preds)

print("Tuned XGBoost (Snowflake ML Tuner):")
for k, v in tuned_metrics.items():
    print(f"  {k}: {v}")

print()
comparison = generate_comparison_report(baseline_metrics, tuned_metrics)
print(comparison.to_string(index=False))

### Visualization — Baseline vs Tuned
Side-by-side metric bars and residual distributions to visually confirm the improvement.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metric_names = ['MAE', 'RMSE', 'MAPE', 'R²']
bvals = [baseline_metrics['mae'], baseline_metrics['rmse'], baseline_metrics['mape'], baseline_metrics['r2']]
tvals = [tuned_metrics['mae'], tuned_metrics['rmse'], tuned_metrics['mape'], tuned_metrics['r2']]

x = np.arange(len(metric_names))
w = 0.35
axes[0].bar(x - w/2, bvals, w, label='Baseline', color='#999999')
axes[0].bar(x + w/2, tvals, w, label='Tuned', color='#29B5E8')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names)
axes[0].legend()
axes[0].set_title('Metrics Comparison', fontsize=13, fontweight='bold')

axes[1].hist(y_test.values - baseline_preds, bins=40, alpha=0.5, color='#999999', label='Baseline', edgecolor='white')
axes[1].hist(y_test.values - tuned_preds, bins=40, alpha=0.5, color='#29B5E8', label='Tuned', edgecolor='white')
axes[1].set_title('Residual Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residual (Actual − Predicted)')
axes[1].legend()

plt.suptitle('Baseline vs Tuned XGBoost (Snowflake ML Tuner)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### Log Tuned Run to Experiment Tracking
Logs the best trial's hyperparameters, metrics, and feature importance to the same experiment as notebook 03, enabling side-by-side comparison of all runs in **Snowsight → AI & ML → Experiments**.

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)
exp.set_experiment(cfg.SNOWHEALTH_EXPERIMENT_NAME)

# Delete stale run if re-running this notebook
try:
    exp.delete_run('xgboost_tuned_sf_tuner')
except:
    pass

best_params = {k.replace('config/', ''): v for k, v in results.best_result.iloc[0].items() if k.startswith('config/')}

with exp.start_run('xgboost_tuned_sf_tuner'):
    exp.log_params(best_params)
    exp.log_param('model_type', 'XGBRegressor_Tuned')
    exp.log_param('hpo_method', 'Snowflake_ML_Tuner_RandomSearch')
    exp.log_param('num_trials', 10)
    exp.log_param('compute_pool', cfg.SNOWHEALTH_COMPUTE_POOL)
    exp.log_metrics(tuned_metrics)

    tuned_fi = get_feature_importance(best_model, feature_cols, 'xgboost')
    from model_utils import save_feature_importance_artifact
    fi_path = save_feature_importance_artifact(tuned_fi, 'xgboost_tuned')
    exp.log_artifact(fi_path, artifact_path='feature_importance')

print('Tuned run logged to experiment tracking')

### Register Tuned Model in Model Registry
Adds the best model as `tuned_v1` alongside the existing `baseline_v1`. Sets the tuned version as the **default** — all downstream `mv.run()` calls without an explicit version will use it.

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model.model_signature import infer_signature

registry = Registry(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)
sig = infer_signature(X_train.head(10), y_train.head(10))

# Drop existing tuned version if re-running (must unset default first)
try:
    m = registry.get_model(cfg.SNOWHEALTH_MODEL_NAME)
    if m.default and m.default.version_name == 'TUNED_V1':
        m.default = 'BASELINE_V1'
    m.delete_version('tuned_v1')
    print('Dropped existing tuned_v1')
except Exception as e:
    print(f'No existing tuned_v1 to drop ({type(e).__name__})')

mv_tuned = registry.log_model(
    best_model,
    model_name=cfg.SNOWHEALTH_MODEL_NAME,
    version_name='tuned_v1',
    signatures={'predict': sig},
    comment=f'SnowHealth demand forecast — HPO tuned via Snowflake ML Tuner (RMSE: {tuned_metrics["rmse"]})',
    metrics=tuned_metrics,
    sample_input_data=X_train.head(10),
    target_platforms=['WAREHOUSE'],
    options={'enable_explainability': True},  # adds explain() for SHAP in notebook 05
)
print(f'Registered: {cfg.SNOWHEALTH_MODEL_NAME}/tuned_v1')

model_ref = registry.get_model(cfg.SNOWHEALTH_MODEL_NAME)
model_ref.default = 'tuned_v1'
print(f'Default version → tuned_v1')

for v in model_ref.versions():
    print(f'  {v.version_name}: {v.comment}')

### Summary
Recap of HPO results, best hyperparameters, and what was registered.

In [ ]:
improvement = baseline_metrics['rmse'] - tuned_metrics['rmse']
print('=' * 60)
print('  HYPERPARAMETER TUNING COMPLETE')
print('=' * 60)
print(f'  Method:        Snowflake ML Tuner (RandomSearch, 10 trials)')
print(f'  Compute pool:  {cfg.SNOWHEALTH_COMPUTE_POOL}')
print(f'  Baseline RMSE: {baseline_metrics["rmse"]}')
print(f'  Tuned RMSE:    {tuned_metrics["rmse"]}')
print(f'  Improvement:   {improvement:.4f} ({improvement/baseline_metrics["rmse"]*100:.1f}%)')
print(f'  Default model: {cfg.SNOWHEALTH_MODEL_NAME}/tuned_v1')
print('=' * 60)
print('  Next → 05_deployment_monitoring.ipynb')